# Bölüm 12: Modelinizi Eğitmek

> "Başarısız olmadım. Sadece işe yaramayacak 10.000 yol buldum."
> — **Thomas Edison**, Mucit

---

## Neler Öğreneceksiniz

- Sonraki token tahminin modellere yazmayı nasıl öğrettiği
- Tüm sinir ağlarına güç veren 5 adımlık eğitim tarifi
- Kayıp eğrilerinin modelinizin öğrenip öğrenmediğini nasıl ortaya çıkardığı
- Aşırı öğrenmeyi nasıl önleyeceğiniz ve ne zaman durmanız gerektiği
- İlerlemenizi asla kaybetmemek için kontrol noktaları kaydetme
- Modelinizin saçmalıktan tutarlılığa dönüşmesini izlemenin heyecanı

---

## Kurulum

İlk olarak, gerekli paketleri yükleyelim:

In [ ]:
# Gerekli paketleri yükle
!pip install -q torch transformers tqdm

In [ ]:
# ===== KÜTÜPHANELER =====
import math
import urllib.request
from functools import partial
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from tqdm import tqdm

# Cihazı ayarla
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan cihaz: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ===== YENİDEN ÜRETİLEBİLİRLİK =====
def set_seed(seed=42):
    """Yeniden üretilebilirlik için tüm tohumları ayarla."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

## 1. Bölüm 10-11'den Model Bileşenleri

İlk olarak, önceki bölümlerde oluşturduğumuz MiniGPT modelini getirelim.

In [ ]:
# ===== ÇOK BAŞLI DİKKAT (Bölüm 10'dan) =====

class MultiHeadAttention(nn.Module):
    """Verimli çok başlı dikkat (tüm başları birlikte yığınlar)."""

    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model, num_heads ile tam bölünebilir olmalı"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape

        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)

        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        attn_output = attn_weights @ V
        attn_output = attn_output.transpose(1, 2).reshape(batch, seq, d_model)

        return self.out_proj(attn_output), attn_weights

print("MultiHeadAttention tanımlandı!")

In [ ]:
# ===== İLERİ BESLEMELI AĞ (Bölüm 10'dan) =====

class FeedForward(nn.Module):
    """Konum bazlı ileri beslemeli ağ."""

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

print("FeedForward tanımlandı!")

In [ ]:
# ===== TRANSFORMER BLOĞU (Bölüm 10'dan) =====

class TransformerBlock(nn.Module):
    """Eksiksiz Transformer bloğu (GPT-2 gibi pre-norm stili)."""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)
        return x, attn_weights

print("TransformerBlock tanımlandı!")

In [ ]:
# ===== GPT YAPILANDIRMASI (Bölüm 11'den) =====

@dataclass
class GPTConfig:
    """MiniGPT modeli için yapılandırma."""
    vocab_size: int = 50257
    max_seq_len: int = 1024
    embed_dim: int = 768
    num_heads: int = 12
    num_layers: int = 12
    d_ff: int = 3072
    dropout: float = 0.1

    def __post_init__(self):
        assert self.embed_dim % self.num_heads == 0, \
            f"embed_dim ({self.embed_dim}), num_heads ({self.num_heads}) ile tam bölünebilir olmalı"

print("GPTConfig tanımlandı!")

In [ ]:
# ===== MİNİGPT MODELİ (Bölüm 11'den) =====

class MiniGPT(nn.Module):
    """Minimal GPT tarzı dil modeli."""

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        # Gömmeler
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # Transformer blokları
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.embed_dim,
                num_heads=config.num_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.num_layers)
        ])

        # Son katman normalizasyonu ve LM başı
        self.ln_f = nn.LayerNorm(config.embed_dim)
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        # Ağırlık bağlama
        self.lm_head.weight = self.token_embed.weight

        # Ağırlıkları başlat
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

    def forward(self, token_ids, return_attention=False):
        batch, seq = token_ids.shape
        device = token_ids.device

        tok_emb = self.token_embed(token_ids)
        positions = torch.arange(seq, device=device)
        pos_emb = self.pos_embed(positions)
        x = self.dropout(tok_emb + pos_emb)

        mask = torch.tril(torch.ones(seq, seq, device=device))

        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, mask)
            if return_attention:
                attention_weights.append(attn)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        if return_attention:
            return logits, attention_weights
        return logits

print("MiniGPT sınıfı tanımlandı!")

## 2. Veri Kümesini İndirme

TinyShakespeare kullanacağız, dakikalar içinde eğitmek için yeterince küçük.

In [ ]:
# TinyShakespeare'i indir
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")

with open("shakespeare.txt", "r") as f:
    text = f.read()

print(f"Veri kümesi boyutu: {len(text):,} karakter")
print(f"\nÖrnek:\n{text[:500]}")

## 3. Veri Kümesi ve DataLoader Oluşturma

In [ ]:
class TextDataset(Dataset):
    """Metin parçaları döndüren basit veri kümesi."""

    def __init__(self, text, chunk_size=256):
        self.chunks = []
        for i in range(0, len(text) - chunk_size, chunk_size):
            self.chunks.append(text[i:i + chunk_size])

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        return self.chunks[idx]


def collate_fn(batch, tokenizer, max_length=128):
    """Bir metin dizesi yığınını tokenize et ve doldur."""
    encoded = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return encoded["input_ids"], encoded["attention_mask"]


# Tokenizer'ı yükle
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Veri kümesini oluştur
dataset = TextDataset(text, chunk_size=256)
print(f"Parça sayısı: {len(dataset)}")

In [ ]:
# Eğitim/doğrulama bölünmesi oluştur
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Eğitim: {len(train_dataset)}, Doğrulama: {len(val_dataset)}")

# DataLoader'ları oluştur
collate = partial(collate_fn, tokenizer=tokenizer, max_length=128)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate
)

# Bir yığını kontrol et
input_ids, attention_mask = next(iter(train_loader))
print(f"Yığın input_ids şekli: {input_ids.shape}")

## 4. Model Oluşturma

In [ ]:
# Hızlı eğitim için küçük yapılandırma
config = GPTConfig(
    vocab_size=50257,
    max_seq_len=128,
    embed_dim=256,
    num_heads=4,
    num_layers=4,
    d_ff=1024,
    dropout=0.1
)

model = MiniGPT(config).to(device)
print(f"Parametreler: {sum(p.numel() for p in model.parameters()):,}")

## 5. "Önce" Durumu: Eğitilmemiş Model

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=30, temperature=1.0):
    """Sıcaklık kontrolü ile metin üret."""
    model.eval()
    device = next(model.parameters()).device

    token_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    for _ in range(max_new_tokens):
        logits = model(token_ids)
        next_logits = logits[:, -1, :] / temperature
        probs = F.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        token_ids = torch.cat([token_ids, next_token], dim=1)

        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(token_ids[0])


# Eğitilmemiş modelden üret
print("EĞİTİMDEN ÖNCE (rastgele ağırlıklar):")
print("="*50)
prompt = "The king"
print(f"İstem: '{prompt}'")
print(f"Çıktı: {generate(model, tokenizer, prompt)}")
print("\n(Rastgele saçmalık, model henüz hiçbir şey öğrenmedi!)")

## 6. Eğitim Kurulumu

In [ ]:
# Eğitim hiperparametreleri
num_epochs = 3
learning_rate = 3e-4
warmup_steps = 100

total_steps = len(train_loader) * num_epochs
print(f"Toplam eğitim adımı: {total_steps}")

# Optimize edici ve zamanlayıcı
optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

## 7. Eğitim ve Değerlendirme Fonksiyonları

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, device, clip_norm=1.0):
    """5 adımlık tarifi kullanarak bir epoch eğitimi."""
    model.train()
    total_loss = 0

    progress = tqdm(dataloader, desc="Eğitim")
    for input_ids, attention_mask in progress:
        input_ids = input_ids.to(device)

        # Dil modellemesi için kaydır
        inputs = input_ids[:, :-1]
        targets = input_ids[:, 1:]

        # ===== 5 ADIMLI TARİF =====
        optimizer.zero_grad(set_to_none=True)       # 1. Gradyanları sıfırla
        logits = model(inputs)                       # 2. İleri geçiş
        loss = F.cross_entropy(                      # 3. Kaybı hesapla
            logits.view(-1, logits.size(-1)),
            targets.reshape(-1),  # reshape, bitişik olmayan dilimleri işler
            ignore_index=tokenizer.pad_token_id
        )
        loss.backward()                              # 4. Geri geçiş
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        optimizer.step()                             # 5. Ağırlıkları güncelle
        scheduler.step()

        total_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(dataloader)


@torch.no_grad()
def evaluate(model, dataloader, device):
    """Modeli değerlendir ve perpleksite hesapla."""
    model.eval()
    total_loss = 0
    total_tokens = 0

    for input_ids, attention_mask in dataloader:
        input_ids = input_ids.to(device)
        inputs = input_ids[:, :-1]
        targets = input_ids[:, 1:]

        logits = model(inputs)
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            targets.reshape(-1),  # reshape, bitişik olmayan dilimleri işler
            ignore_index=tokenizer.pad_token_id,
            reduction='sum'
        )

        mask = (targets != tokenizer.pad_token_id)
        total_loss += loss.item()
        total_tokens += mask.sum().item()

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    return avg_loss, perplexity

In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, train_loss, val_loss, path):
    """Eğitim kontrol noktasını kaydet."""
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
    }, path)
    print(f"Kontrol noktası kaydedildi: {path}")

## 8. Eğitim Döngüsü

In [ ]:
# Eğitim!
train_losses = []
val_losses = []
best_val_loss = float('inf')

for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"{'='*50}")

    # Eğit
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    train_losses.append(train_loss)

    # Değerlendir
    val_loss, perplexity = evaluate(model, val_loader, device)
    val_losses.append(val_loss)

    print(f"\nEğitim Kaybı: {train_loss:.4f}")
    print(f"Doğrulama Kaybı:   {val_loss:.4f}")
    print(f"Perpleksite: {perplexity:.1f}")

    # En iyi kontrol noktasını kaydet
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model, optimizer, scheduler, epoch,
                       train_loss, val_loss, "best_model.pt")

print("\nEğitim tamamlandı!")
print(f"En iyi doğrulama kaybı: {best_val_loss:.4f}")

## 9. Kayıp Eğrilerini Çizme

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
epochs = range(1, len(train_losses) + 1)
plt.plot(epochs, train_losses, 'b-o', label='Eğitim Kaybı')
plt.plot(epochs, val_losses, 'r-o', label='Doğrulama Kaybı')
plt.xlabel('Epoch')
plt.ylabel('Kayıp')
plt.title('Eğitim ve Doğrulama Kaybı')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 10. Ödül: Metin Üretimi!

In [ ]:
# En iyi modeli yükle
checkpoint = torch.load("best_model.pt", map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Kontrol noktası {checkpoint['epoch'] + 1}. epoch'tan yüklendi")
print(f"Doğrulama kaybı: {checkpoint['val_loss']:.4f}")

In [ ]:
print("\n" + "="*60)
print("EĞİTİMDEN SONRA (Shakespeare üzerinde):")
print("="*60)

prompts = [
    "The king",
    "To be or not to be",
    "Friends, Romans, countrymen",
    "All the world's a stage"
]

for prompt in prompts:
    output = generate(model, tokenizer, prompt, max_new_tokens=40, temperature=0.8)
    print(f"\nİstem: '{prompt}'")
    print(f"Çıktı: {output}")
    print("-"*40)

## 11. Önce ve Sonra Karşılaştırması

In [ ]:
# Karşılaştırma için yeni eğitilmemiş model oluştur
set_seed(123)  # Farklı rastgele ağırlıklar için farklı tohum
untrained_model = MiniGPT(config).to(device)
untrained_model.eval()

prompt = "The fair maiden"

print("="*60)
print(f"İstem: '{prompt}'")
print("="*60)
print("\nÖNCE (eğitilmemiş):")
print(generate(untrained_model, tokenizer, prompt, max_new_tokens=30))
print("\nSONRA (Shakespeare üzerinde eğitilmiş):")
print(generate(model, tokenizer, prompt, max_new_tokens=30, temperature=0.8))
print("\nAynı mimari. Aynı kod. Eğitim tüm farkı yaratıyor!")

## 12. Sıcaklık Karşılaştırması

In [ ]:
prompt = "The noble lord"

print(f"İstem: '{prompt}'\n")
for temp in [0.5, 0.8, 1.0, 1.5]:
    output = generate(model, tokenizer, prompt, max_new_tokens=30, temperature=temp)
    print(f"Sıcaklık {temp}:")
    print(f"  {output}")
    print()

## Özet

**Neler oluşturduk:**

1. Sonraki token tahmini için **etiket kaydırma**
2. Metni verimli bir şekilde yığınlayan ve tokenize eden **DataLoader'lar**
3. **5 adımlık eğitim tarifi**: zero_grad → forward → loss → backward → step
4. Doğrulama kaybı ve perpleksite ile **değerlendirme**
5. Eğitimi kaydetmek ve devam ettirmek için **kontrol noktası oluşturma**
6. Sıcaklık kontrolü ile **metin üretimi**

**Temel kavramlar:**

- Eğitim bir geri besleme kontrol döngüsüdür: hatayı ölç, ağırlıkları ayarla, tekrarla
- Çapraz entropi kaybı, belirsiz olanlardan daha fazla kendinden emin yanlış tahminleri cezalandırır
- Aşırı öğrenme = eğitim verisini ezberlemek (eğitim kaybı ↓, doğrulama kaybı ↑)
- Perpleksite, modelin "ne kadar şaşırdığını" ölçer, düşük olanı daha iyidir
- Sıcaklık, üretim çeşitliliğini kontrol eder

**Sonrası:** Bölüm 13, bu modeli belirli görevler için nasıl ince ayarlayacağınızı öğretecek!

## Alıştırmalar

### Alıştırma 1: Öğrenme Oranı Deneyi

Farklı öğrenme oranlarını deneyin ve sonuçları karşılaştırın.

In [ ]:
# KODUNUZ BURAYA
# lr=1e-5, lr=3e-4, lr=1e-2 ile eğitin
# Kayıp eğrilerini karşılaştırın
# Hangi öğrenme oranı en iyi çalışıyor?

### Alıştırma 2: Daha Fazla Epoch

In [ ]:
# KODUNUZ BURAYA
# 3 yerine 5-10 epoch için eğitin
# Model gelişmeye devam ediyor mu?
# Aşırı öğrenme belirtileri görüyor musunuz?

### Alıştırma 3: Model Boyutu Karşılaştırması

In [ ]:
# KODUNUZ BURAYA
# Daha küçük bir model oluşturun (2 katman, 128 boyut)
# Daha büyük bir model oluşturun (6 katman, 384 boyut)
# Eğitim hızını ve nihai perpleksiteyi karşılaştırın